# NumPy y SciPy, entrada/salida con H5Py

### Contenido
* NumPy
* Ejemplos de SciPy
* Entrada/salida con arreglos de NumPy
* Bonus: ejemplos de procesamiento de imágenes

## El problema de rendimiento: los loops

* Iterar (`for`) en Python, como en la mayoría de los lenguajes interpretados, puede ser **muy costoso**.
* El cómputo numérico casi siempre consiste en operaciones sobre arreglos completos, no en procesar un número a la vez.
* Ejemplo: la Transformada Discreta de Fourier (DFT), que descompone una señal en las frecuencias que la componen:

$$ X_k = \sum_{n=0}^{N-1}x_n\cdot e^{\frac{-2i\pi}{N}kn} $$

No hace falta que entiendas la fórmula en detalle — lo que importa es que es una suma que se repite para cada uno de los $N$ valores de salida, es decir, $O(N^2)$ operaciones. Vamos a implementarla de tres formas distintas y comparar.

In [ ]:
import cmath
import random

def DFT_loop(x):
    # version con loops explicitos: una suma por cada frecuencia k
    N = len(x)
    X = [0 for n in range(N)]
    for k in range(N):
        for n in range(N):
            X[k] += x[n]*cmath.exp(-2j * cmath.pi * k * n / N)
    return X

X = [random.random() for i in range(512)]
F = DFT_loop(X)

### Solución: arreglos de NumPy

Convertimos los coeficientes del loop en arreglos y usamos operaciones de álgebra lineal (multiplicación de matrices) en vez de loops explícitos. El costo: usa más memoria (construye una matriz de $N \times N$ completa). El beneficio: mucho más rápido que la versión con loops.

$$ X_k = \sum_{n=0}^{N-1}x_n\cdot e^{\frac{-2i\pi}{N}kn} $$

In [ ]:
import numpy as np

def DFT_arrays(x):
    x = np.asarray(x, dtype=float)
    N = x.shape[0]
    n = np.arange(N)
    k = n.reshape((N, 1))
    M = np.exp(-2j * np.pi * k * n / N)
    return np.dot(M, x)

X = np.random.random(512)
F = DFT_arrays(X)

### Mejor solución: llamar a una librería externa

* Las librerías externas de bajo nivel (C, Fortran) casi siempre van a ser *mucho* mejores que reimplementar el algoritmo a mano.
* Llaman a funciones ya optimizadas que usan OpenMP, aciertan bien en caché, usan SIMD, etc. — todo lo que viste en el notebook de "Bases de HPC".
* NumPy y SciPy ya traen implementaciones de FFT (y muchas otras operaciones) que delegan a esas librerías.

**Actividad:** ejecuta la siguiente celda y compara los tres tiempos.

In [ ]:
X = np.random.random(512)

%timeit Fl = DFT_loop(X)
%timeit Fa = DFT_arrays(X)
%timeit Fe = np.fft.fft(X)

**Tus resultados:** anota los tres tiempos y compáralos. ¿Coincide con el patrón que viste en el teaser de difusión (loops < NumPy < librería optimizada)?

## El pan de cada día: NumPy y SciPy

* **NumPy:**
  * Maneja eficientemente operaciones sobre arreglos mediante el objeto `ndarray`.
  * Ejecuta los loops en código C ya compilado — mucho más rápido.
* **SciPy:**
  * Rutinas matemáticas y numéricas más avanzadas, construidas sobre NumPy.
  * Ofrece varias formas de trabajar con matrices dispersas ("sparse").
* Ambos, en el fondo, llaman a las mismas librerías estándar de álgebra lineal: BLAS y LAPACK (mediante `numpy.linalg` y `scipy.linalg`).

## Arreglos de NumPy

El [`ndarray`](https://numpy.org/doc/stable/reference/arrays.ndarray.html) (*N-dimensional array*) es la clase fundamental de NumPy para guardar datos:

* Guarda elementos del mismo tipo y tamaño (a diferencia de una lista de Python, que puede mezclar tipos).
* Por dentro, es literalmente un arreglo de C, con los valores guardados de forma lineal en memoria (recuerda la sección de memoria del notebook 03).
* Conserva una sintaxis parecida a la de Python, con muchísimos métodos para operaciones matemáticas.

Es una gran oportunidad de rendimiento porque:
* Puede aprovechar características de la CPU como *prefetching*, caché y SIMD (vectorización).
* Puede llamar por dentro a librerías de altísimo rendimiento como MKL (Intel), LAPACK o BLAS.

In [ ]:
import numpy as np
my_array = np.array([[1,2,3,4], [5,6,7,8]])
print(my_array)
print(type(my_array))

### Constructores de arreglos

* Muchos constructores piden una tupla de "forma" (`shape`) — los paréntesis son obligatorios.
* Aceptan cualquier número de dimensiones.

In [ ]:
x = np.zeros((3,4))
x = np.ones((2,3,4))
x = np.full((2,2), 7.2)

# arreglos aleatorios
x = np.random.random((2,2))      # matriz 2x2 de reales aleatorios
x = np.random.randint(10, size=5) # 5 enteros aleatorios entre 0 y 9

# memoria sin inicializar (mas rapido de crear, pero con "basura" adentro)
x = np.empty((3,2))

# inicio, fin, paso
x = np.arange(10, 55, 5)
print("Rango discreto:", x)

# primer elemento, ultimo elemento, cantidad total de elementos
x = np.linspace(0, 1, 9)
print("Rango continuo:", x)

**Actividad:** en la celda de abajo, crea tú mismo:
1. Un arreglo de ceros de forma `(4, 5)`.
2. Un arreglo con los números pares del 0 al 20 (usando `np.arange`).
3. Un arreglo de 6 números igualmente espaciados entre -1 y 1 (usando `np.linspace`).

Para cada uno, imprime su `shape` y su `dtype`.

In [ ]:
# Actividad: completa aqui tu codigo
# 1. arreglo de ceros de forma (4, 5)

# 2. numeros pares del 0 al 20

# 3. 6 numeros entre -1 y 1


### Tipos de datos y compatibilidad

* Los tipos de NumPy corresponden a tipos nativos de C.
* Se puede especificar el tipo con el parámetro `dtype=` del constructor.
* Las operaciones convierten automáticamente entre tipos compatibles.
* Los valores por defecto (en la mayoría de sistemas Linux/x86_64) son `np.float64` y `np.int64`.

In [ ]:
import numpy as np
x = np.array([1,2,3])
y = np.array([1.,2.,3.])
print("Tipo de x y de y: %s y %s" % (x.dtype, y.dtype))

r = x+y
print("Tipo de x+y: %s" % (r.dtype))

z = np.array([1+2j,2+3j,3+4j])
sz = z.astype(np.complex64)
print("Tipo de z y de sz: %s y %s" % (z.dtype, sz.dtype))

**Actividad — predice antes de ejecutar:** ¿qué `dtype` esperas que tenga el resultado de `np.array([1, 2, 3]) + np.array([1.5, 2.5, 3.5])`? Anota tu predicción, y después ejecuta la celda para comprobarla.

Predicción: ___

In [ ]:
resultado = np.array([1, 2, 3]) + np.array([1.5, 2.5, 3.5])
print(resultado.dtype)

### Funciones sobre arreglos

Los arreglos tienen una enorme cantidad de operaciones predefinidas — demasiadas para listarlas todas aquí.

In [ ]:
x = np.random.randint(10, size=8)
print("* Arreglo sin ordenar: ", x)
x.sort()
print("* Arreglo ordenado: ", x)

# calcular algunas propiedades
y = np.random.random((10,3))
print("* Max, min, media, desv. estandar, suma: ", y.max(), y.min(), y.mean(), y.std(), y.sum())

# tambien se puede especificar sobre que eje calcular
print("* Maximo de cada columna: ", y.max(axis=0))
print("* Maximo de cada fila: ", y.max(axis=1))

# comparar dos arreglos
z = y + 1e-16
print("* Iguales exactamente: ", np.array_equal(y,z))
print("* Iguales aproximadamente: ", np.allclose(y, z, 1e-8))

### Rendimiento de las funciones de arreglos

Regla práctica: prefiere siempre usar las funciones *ya integradas* de NumPy en vez de reimplementarlas con un loop — son más rápidas y el código queda más limpio.

In [ ]:
# comparacion: loop propio vs. funcion de numpy
def mean_loop(a):
    result = 0
    for i in range(len(a)):
        result += a[i]
    return result/len(a)

x = np.arange(2**20)

%timeit mean_loop(x)
%timeit np.mean(x)

**Tus resultados**

- Tiempo de `mean_loop(x)`: ___
- Tiempo de `np.mean(x)`: ___
- ¿Cuántas veces más rápida fue la función integrada de NumPy?

### Atributos de los arreglos

Son un poco más avanzados, pero útiles si vas a escribir interfaces entre Python y C/Fortran (notebook 06), o si necesitas verificar que los datos son contiguos en memoria (recuerda la jerarquía de memoria del notebook 03).

In [ ]:
print(y.ndim)     # numero de dimensiones
print(y.size)     # numero total de elementos
print(y.dtype)    # tipo de dato
print(y.flags)    # informacion sobre la disposicion en memoria
print(y.itemsize) # tamano de un elemento, en bytes
print(y.nbytes)   # tamano total del arreglo, en bytes

**Actividad:** crea una vista con paso distinto de 1 (por ejemplo `v = y[::2]`) e imprime `v.flags`. Busca la línea `C_CONTIGUOUS` — ¿dice `True` o `False`? Relaciona esto con lo que viste sobre acceso lineal a memoria en el notebook 03: una vista no contigua puede ser más lenta de procesar.

In [ ]:
v = y[::2]
print(v.flags)

## Indexado de arreglos

* La sintaxis es idéntica a la de Python estándar: `x[sel]`.
* **No** garantiza que el resultado sea contiguo en memoria.
* El indexado multidimensional `x[sel1, sel2, sel3]` es una forma corta de `x[(sel1, sel2, sel3)]`.

### Indexado básico

* Indexar un solo elemento:
  * `x[0]`
  * `x[0,1]` (equivalente a `x[0][1]`, pero más eficiente)
* *Slicing* (rebanado) con paso: el objeto de selección es un `slice` (`inicio:fin:paso`), o una tupla de `slice`s, enteros, `newaxis`, o `Ellipsis`.
* El indexado básico generalmente crea una **vista** del arreglo original (más sobre esto abajo).

In [ ]:
x = np.arange(20)

# el slicing basico es igual que en listas de Python: inicio:fin:paso
# para multiples dimensiones: [inicio1:fin1:paso1, inicio2:fin2:paso2, ...]
print("Del 2 al 9, en pasos de 2:", x[2:9:2])
print("Todos menos los ultimos 3:", x[:-3])

# invertir el arreglo
print("Al reves:", x[::-1])

# convertir a un arreglo multidimensional
y = x.reshape((5,4))
print("Forma de y:", y.shape)
print("Un slice de y:", y[1:,:2])

# se pueden agregar nuevas dimensiones
z = np.arange(3)
print("z:")
print(z[:,np.newaxis])

### Broadcasting

*Broadcasting* es el mecanismo por el cual NumPy permite operar arreglos de formas distintas, "expandiendo" automáticamente el más chico para que encaje con el más grande — sin copiar datos de verdad. Algunas conversiones de dimensión son automáticas, otras no lo son, así que hay que tener cuidado con la legibilidad del código.

![broadcasting](fig/numpy_broadcast.svg)

In [ ]:
X = np.arange(12).reshape(4,3)
Y = np.arange(3)/10
Z = np.arange(4)/10
print("X es una matriz 4x3:\n", X)

print("\nX+Y es valido (Y tiene 3 elementos, coincide con las columnas de X):\n", X+Y)

# X+Z no es valido directamente! Z tiene 4 elementos (coincide con las filas, no columnas)
# hace falta agregar una nueva dimension para que el broadcasting sepa que Z va por filas:
print("\nX+Z[:,None] es valido:\n", X+Z[:,None])

**Regla de broadcasting, en corto:** NumPy compara las formas de los dos arreglos de derecha a izquierda; dos dimensiones son compatibles si son iguales, o si una de las dos es 1. `Z[:,None]` convierte a `Z` de forma `(4,)` a forma `(4,1)`, para que se pueda "estirar" a lo largo de las columnas de `X`. Si esto no queda del todo claro todavía, no te preocupes — es de los conceptos de NumPy que más cuesta interiorizar; vuelve a esta celda cuando te encuentres un error de "shapes not aligned" en tu propio código.

**Actividad:** ejecuta `X + Z` (sin el `[:,None]`) en la celda de abajo. Va a fallar — eso es intencional. Lee el mensaje de error (`ValueError: operands could not be broadcast together with shapes ...`) y relaciónalo con la regla que acabas de leer: ¿qué formas está comparando NumPy, y por qué no son compatibles?

In [ ]:
# Esta celda deberia fallar - es intencional, para que leas el mensaje de error
X + Z

### Indexado avanzado

El objeto de selección (en `x[obj]`) también puede ser:
* Una secuencia que no es tupla (una lista o un `range`), o un `ndarray` de índices.
* Una tupla que contenga al menos una secuencia o `ndarray` de ese tipo.
* Un arreglo booleano (`True`/`False`) — debe tener la misma forma que el arreglo indexado.

In [ ]:
x = np.arange(10)*10
print(x)

# usar una lista de enteros como indices
print("x con indices enteros: ", x[[2,5,8]])
print("x con indices booleanos: ", x[[False, False, True, False, False, True, False, False, True, False]])

# crear un arreglo booleano
y = x**2 - 750
index = (x < y)
print(index)
print("x con indices booleanos: ", x[index])
print("x con indices negados: ", x[~index])
print("x en una sola linea: ", x[x<y])

# asignar valores usando indexado booleano
# x[x < y] = 100 es equivalente a:
# for i in range(len(x)):
#     if y[i] < 10:
#         x[i] = 100
x[x < y] = 100
print("x modificado en una linea: ", x)

## Arreglos vs. vistas

* El *slicing* básico normalmente devuelve una **vista** del arreglo original, no una copia.
* Los datos **no** se copian $\to$ si cambias el arreglo original, la vista también cambia (y viceversa).
* Las vistas pueden tener datos **no contiguos** en memoria (por ejemplo, si usaste un `paso` distinto de 1).
* Usa `.copy()` cuando de verdad necesites una copia independiente de los datos.
* El indexado avanzado (con listas o booleanos), en cambio, **siempre** devuelve una copia.

Esto puede ser una fuente de bugs difíciles de encontrar si no lo tienes presente: modificar una "vista" modifica silenciosamente el arreglo original.

In [ ]:
x = np.arange(15)
print("Arreglo x: ", x)
y = x[1:8:2]
print("Vista y: ", y)
print(y.flags)
x[1] = 20
print("Vista y despues de cambiar x: ", y)
y[0] *= 10
print("x original despues de cambiar la vista y: ", x)

c = y.copy()
print("Copia de y: ", c)
print(c.flags)

### Vistas vs. copias: rendimiento

* Acceder a datos con "paso" (*strided*) puede perjudicar el rendimiento (memoria no contigua $\to$ peor uso de la caché, ver notebook 03).
* Usar un arreglo temporal (una copia) puede ser preferible en ese caso — pero copiar también tiene un costo.
* Ten cuidado al pasar vistas como argumento a una función: quien la recibe puede no darse cuenta de que está compartiendo memoria con el original.

In [ ]:
import numpy as np

def sum_test(c,x,y):
    # calcula c*x^2 + y
    if len(c) == len(x) == len(y):
        return c*x*x + y
    else:
        print("Forma incorrecta")
        return None

def sum_test_copy(c,x,y):
    copia = x.copy()
    return sum_test(c,copia,y)

x = np.random.random(1_000_000)
y = np.random.random(100_000)
c = np.random.random(100_000)
vista = x[::10]        # vista con paso 10: datos NO contiguos
copia = vista.copy()   # copia de esos mismos datos, SI contiguos

%timeit res = sum_test(c,vista,y)
%timeit res = sum_test(c,copia,y)
%timeit res = sum_test_copy(c,vista,y)

**Tus resultados**

- Tiempo usando la vista directamente: ___
- Tiempo usando la copia ya hecha de antemano: ___
- Tiempo copiando dentro de la función: ___

**Para pensar:** ¿por qué la tercera versión (que copia *dentro* de la función) puede terminar siendo más lenta que simplemente usar la vista, a pesar de que opera sobre datos contiguos? (pista: el costo de hacer la copia hay que sumarlo al tiempo total).

## Ejemplo 1: cálculo de puntos medios

$$y_i = \dfrac{x_{i}+x_{i+1}}{2}, \qquad \forall i<N$$

Tres implementaciones del mismo cálculo, de la peor a la mejor:

In [ ]:
def midpoints_list(x):    # la peor implementacion
    y = []
    for i in range(len(x)-1):
        y.append(0.5*(x[i] + x[i+1]))
    return np.array(y)

def midpoints_loop(x):    # loop, pero forzando un arreglo de numpy
    N = len(x)
    y = np.zeros(N-1)
    for i in range(N-1):
        y[i] = 0.5*(x[i] + x[i+1])
    return y

def midpoints_array(x):   # la mejor: sin ningun loop
    return 0.5*(x[:-1] + x[1:])

x = np.linspace(0, 10, 10**6)
%timeit midpoints_list(x)
%timeit midpoints_loop(x)
%timeit midpoints_array(x)

**Tus resultados**

- Tiempo con lista de Python: ___
- Tiempo con loop + arreglo de NumPy: ___
- Tiempo vectorizado (sin loops): ___
- ¿La diferencia entre la primera y la segunda versión te sorprende? Ambas usan un loop explícito — la diferencia está solo en si el resultado se va guardando en una lista o en un arreglo de NumPy ya reservado.

### Cómo se traduce el loop a slicing

$$y_i = \dfrac{x_{i}+x_{i+1}}{2}, \qquad \forall i<N$$

La receta general para convertir un loop en operaciones de arreglos:
* El loop `for i in range(N-1):` se traduce al lado izquierdo como `y[0:N-1]`.
* Los índices del lado derecho se traducen a rangos: `x[i]` se convierte en `x[0:N-1]`, y `x[i+1]` en `x[1:N]`.

![indexing](fig/numpy-indexing.svg)

## Ejemplo 2: manipulaciones "en el lugar" (in-place)

`x[1:] + x[:-1]` crea un arreglo temporal nuevo en memoria. Podemos evitarlo modificando los datos "en el lugar" — pero, ¿siempre vale la pena?

In [ ]:
def temp_arrays(x):
    return 0.5*(x[1:] + x[:-1])

def in_place(x):
    y = x[1:]
    y += x[:-1]   # modifica y "en el lugar", sin crear un arreglo temporal nuevo
    y *= 0.5
    return y

x = np.random.rand(2**15)
%timeit temp_arrays(x)
%timeit in_place(x)

**Tus resultados**

- Tiempo con arreglo temporal: ___
- Tiempo con operación "en el lugar": ___

* El rendimiento depende del problema específico y del tamaño de los datos.
* Puede ser más rápido, pero no siempre lo es — mide antes de asumir.
* Ten presente que `in_place` modifica `y` (que es una vista de `x[1:]`), así que ten cuidado con la mutabilidad si `x` se usa después en otro lado.

## Ejemplo 3: centro de masa

Para posiciones $\vec{x}_i$ (coordenada $j$-ésima $x_{i,j}$) y masas $m_i$:

$$ \vec{x}_\mathsf{com} = \frac{1}{M} \sum_{i=0}^N m_i \vec{x}_i, \qquad\text{donde}\qquad M=\sum_i m_i$$

Tres implementaciones, cada una más "vectorizada" que la anterior:

In [ ]:
import numpy as np
def center_of_mass_loop(pos, mass):
    com = np.zeros(3)
    for j in range(3):
        for i in range(pos.shape[0]):
            com[j] += pos[i, j] * mass[i]
    total_mass = 0.0
    for i in range(pos.shape[0]):
        total_mass += mass[i]
    for j in range(3):
        com[j] /= total_mass
    return com

def center_of_mass_arrays(pos, mass):
    com = np.zeros(3)
    for j in range(3):
        com[j] = np.sum(pos[:, j]*mass[:])
    total_mass = np.sum(mass)
    com /= total_mass
    return com

def center_of_mass_dot(pos, mass):
    # el producto punto (dot) hace toda la suma ponderada de una vez
    return np.dot(mass, pos)/np.sum(mass)

# posiciones y masas de particulas
N = 10**5
pos = np.random.rand(N, 3)
mass = np.random.rand(N)

%timeit center_of_mass_loop(pos, mass)
%timeit center_of_mass_arrays(pos, mass)
%timeit center_of_mass_dot(pos, mass)

**Tus resultados**

- Tiempo con loops explícitos: ___
- Tiempo con `np.sum` por columna: ___
- Tiempo con `np.dot`: ___
- De las tres, `center_of_mass_dot` es la que menos código tiene. ¿También es la más rápida en tu máquina? ¿Por qué crees que pasa eso (piensa en qué hace `np.dot` por debajo)?

## Resumen de NumPy

Mensajes clave para quedarte:

1. Las manipulaciones de alto nivel (organizar el flujo del programa) se hacen en Python.
2. El cómputo intensivo se delega a las librerías que hay por debajo de NumPy.
3. Si no existe una librería dedicada para tu problema, usa arreglos de NumPy y sus funciones — evita loops explícitos sobre elementos individuales.

El costo de esto: hay un compromiso entre productividad de desarrollo y rendimiento. El código en NumPy *puede* ser tan rápido como C o Fortran, pero frecuentemente es entre 2 y 4 veces más lento. Si necesitas más, el camino a seguir es implementar los "puntos calientes" (*hotspots*) en C/Fortran con una interfaz a Python, o usar compilación JIT (Numba, JAX) — ambos temas de notebooks más adelante en el curso.

# SciPy

SciPy es una caja de herramientas matemática construida sobre NumPy, que ofrece funciones para:
* FFT (transformadas de Fourier)
* Integración numérica
* Interpolación
* Álgebra lineal
* Operaciones con matrices dispersas ("sparse")
* Optimización
* Funciones especiales (por ejemplo, funciones de Bessel)
* ... y muchas más

La mayoría delega el trabajo pesado a librerías de C/Fortran, igual que NumPy. Ver la [documentación completa](https://docs.scipy.org/doc/scipy/reference/).

## SciPy - Ejemplo 1: FFT y espectro de potencia

Calculamos y graficamos el espectro de potencia de una señal (código original adaptado del tutorial de SciPy).

In [ ]:
import numpy as np
from scipy.fft import fft, fftfreq
import matplotlib.pyplot as plt
N = 600         # numero de puntos de muestra
T = 1.0 / 800.0 # espaciado entre muestras
x = np.linspace(0.0, N*T, N)
y = np.sin(50.0 * 2.0*np.pi*x) + 0.5*np.sin(80.0 * 2.0*np.pi*x)

yf = fft(y)
xf = fftfreq(N, T)

plt.plot(xf[0:N//2], 2.0/N * np.abs(yf[0:N//2]))
plt.grid()

La señal original es la suma de dos ondas seno (de 50 Hz y 80 Hz). El gráfico del espectro de potencia debería mostrarte justamente dos picos, en esas dos frecuencias — la FFT "descompone" la señal y te dice qué frecuencias la componen.

**Actividad:** vuelve a la celda del ejemplo de FFT (arriba) y cambia las frecuencias de las dos ondas seno (los valores `50.0` y `80.0`) por otras dos de tu elección (por ejemplo, `30.0` y `120.0`). Vuelve a ejecutar la celda. ¿Los picos del gráfico aparecen donde esperabas?

## SciPy - Ejemplo 2: integración numérica

Integración numérica de:
* Una función dada como objeto.
* Una muestra de valores de una función.
* EDOs (ecuaciones diferenciales ordinarias).

In [ ]:
import scipy.integrate as integrate
resultado = integrate.quad(lambda x: x**2, 0, 2.5) # usa QUADPACK de Fortran por debajo
print(resultado[0], 2.5**3/3.)

x = np.linspace(0, 2.0, 10)
y = np.sin(x)
resultado = integrate.simpson(y, x=x) # regla de Simpson para una muestra dada
print(resultado, 1.-np.cos(2.0))

# derivada simple: exponencial
def yprime(y, t, alpha):
    return np.exp(alpha*t)*alpha

ts = np.linspace(0, 2.0, 50)
y0 = 1.0
y = integrate.odeint(yprime, y0, ts, args=(2.0,))  # usa ODEPACK de Fortran por debajo
print(y[-1], np.exp(2.0 * ts[-1])) # comparar con el resultado analitico

En los tres casos, comparamos el resultado numérico contra el resultado analítico exacto (calculado a mano) para verificar que coinciden — una buena práctica siempre que puedas hacerlo.

## SciPy - Ejemplo 3: sistemas lineales dispersos vs. densos

* Denso: `scipy.linalg`. Disperso ("sparse"): `scipy.sparse.linalg`.
* Una matriz **dispersa** es una matriz donde la mayoría de los elementos son cero. En vez de guardar todos esos ceros, se guardan solo las posiciones y valores distintos de cero — mucho más eficiente en memoria y en cómputo, si la matriz es lo bastante dispersa.
* Resolver un sistema lineal con una matriz dispersa puede ser mucho más rápido que con una matriz densa, dependiendo de qué tan dispersa (qué porcentaje de ceros) sea.

In [ ]:
from scipy import sparse
from scipy import linalg
import scipy.sparse.linalg as splinalg
import numpy as np

def comparar_densidad(densidad, n=1000):
    b = np.random.random(n)
    A1 = sparse.random(n, n, density=densidad, format='csr')
    A1 += sparse.eye(n,n, format='csr')
    print("Densidad: ", densidad)
    print("Dispersa (sparse): ")
    %timeit splinalg.spsolve(A1, b)
    A2 = A1.todense()
    print("Densa (dense): ")
    %timeit linalg.solve(A2, b)

# Densidad=0.01: la matriz densa gana. Densidad=0.001: la dispersa gana.
comparar_densidad(0.01)
comparar_densidad(0.001)

**Tus resultados**

- Con densidad 0.01 — tiempo disperso: ___ / tiempo denso: ___ / ¿cuál ganó?
- Con densidad 0.001 — tiempo disperso: ___ / tiempo denso: ___ / ¿cuál ganó?

**Para pensar:** ¿por qué crees que con densidad muy baja (pocos elementos distintos de cero) gana la versión dispersa, pero con densidad un poco más alta gana la densa? (pista: guardar y recorrer la estructura "sparse" también tiene un costo fijo, que solo vale la pena si hay muy pocos elementos que procesar de verdad).

## SciPy - Ejemplo 4: interpolación

Interpolación de datos en 1D y en varias dimensiones (sobre grilla estructurada o puntos sin estructura), usando splines u otros polinomios.

In [ ]:
import scipy.interpolate as interpolate
x = np.linspace(0, 10.0, 12)
y = np.sin(x)
y_int1 = interpolate.interp1d(x, y)
y_int2 = interpolate.PchipInterpolator(x, y)

import matplotlib.pyplot as plt
plt.figure(figsize=(20,6))
plt.subplot(1,2,1)
plt.plot(x, y)
x2 = np.linspace(0, 10.0, 50)
plt.plot(x2, y_int1(x2))

plt.subplot(1,2,2)
plt.plot(x, y)
im = plt.plot(x2, y_int2(x2))

## Resumen de NumPy/SciPy

* **NumPy:** manejo eficiente de arreglos con el objeto fundamental `ndarray` y sus métodos.
  * Evita los loops explícitos elemento por elemento; en su lugar, expresa el cálculo con operaciones de arreglos.
  * Existen otras implementaciones de `ndarray` para casos especiales: GPU (CuPy, JAX), paralelización distribuida (Dask), o entrada/salida (h5py) — los verás más adelante en el curso.
* **SciPy:** rutinas matemáticas y numéricas más avanzadas; usa NumPy por debajo, además de otras librerías de C/Fortran.
* Consejos de rendimiento:
  * Trabaja sobre arreglos completos (slicing, funciones de NumPy...).
  * Identifica los "puntos calientes" (*hotspots*) de tu código y optimiza específicamente esos — perfilar antes de optimizar (notebook 07).
  * Si necesitas más velocidad: código en Cython, o C/Fortran con interfaz a Python, o compilación JIT con Numba o JAX.

## Para seguir leyendo

* Harris, C.R., Millman, K.J., van der Walt, S.J. et al. *Array programming with NumPy.* **Nature** 585, 357–362 (2020). (https://doi.org/10.1038/s41586-020-2649-2)
* Bressert, E. (2012). *SciPy and NumPy* (1ra edición). O'Reilly Media, Inc.
* Documentación oficial de [NumPy](https://numpy.org/doc/stable/) y [SciPy](https://docs.scipy.org/doc/scipy/reference/).

# Entrada y salida con NumPy

## Leer y escribir arreglos de NumPy

Dos posibilidades:
* Las funciones nativas de entrada/salida de NumPy.
* HDF5, usando H5Py.

## Entrada/salida nativa de NumPy

### Guardar arreglos de NumPy en archivos

In [ ]:
import os
import numpy as np

x = np.linspace(0.0,100.0,101)

# guardar como archivo de texto (solo para depurar o archivos chicos: es ineficiente)
np.savetxt("x.txt", x)
print("savetxt: ", os.path.getsize('x.txt'), "bytes")

# guardar como archivo binario, extension ".npy"
np.save("x", x)
print("save: ", os.path.getsize('x.npy'), "bytes")

y = np.linspace(0.0,100.0,201)
# guardar varios arreglos en un archivo binario, extension ".npz", conservando los nombres
np.savez("xy", x=x, y=y)
print("savez: ", os.path.getsize('xy.npz'), "bytes")

np.savez_compressed("xy_compressed", x=x, y=y)
print("savez_compressed: ", os.path.getsize('xy_compressed.npz'), "bytes")

Compara los tamaños en bytes que imprime la celda: el archivo de texto (`savetxt`) ocupa notablemente más que el binario (`save`), y la versión comprimida (`savez_compressed`) es la más chica de todas — a costa de un poco más de tiempo de cómputo al guardar y leer.

**Tus resultados**

- Tamaño de `x.txt` (texto): ___ bytes
- Tamaño de `x.npy` (binario): ___ bytes
- Tamaño de `xy_compressed.npz` (comprimido): ___ bytes
- ¿Cuántas veces más grande es el archivo de texto comparado con el binario comprimido?

### Leer arreglos de NumPy desde archivos

In [ ]:
import numpy as np

#x = np.loadtxt("x.txt")
x = np.load("x.npy")
print("tipo de x:", type(x))
xy = np.load("xy.npz")
print("tipo de xy:", type(xy))

# los arreglos individuales dentro de xy son accesibles con sintaxis de diccionario
print("Claves de xy: ", tuple(xy.keys()))
print("x y xy['x'] son iguales: ", np.allclose(x, xy["x"]))
print("y y xy['y'] son iguales: ", np.allclose(y, xy["y"]))

### ¿Cuándo usar la I/O nativa de NumPy?

* Útil para:
  * Guardar resultados rápidamente (por ejemplo, desde una sesión interactiva).
  * Cachear localmente resultados de cálculos costosos.
  * Datos que siempre se van a leer de la misma forma, desde el mismo tipo de programa.
* A favor: portabilidad binaria entre sistemas (independiente del *endianness* de la máquina).
* En contra: no es fácilmente accesible desde otros programas o lenguajes — si otra persona con MATLAB o C++ necesita tus datos, un `.npy` no le sirve de mucho.

## HDF5 con H5Py

### HDF5 en pocas palabras

* HDF5 es un formato de datos jerárquico, con su propia especificación e implementación de librería.
* Contiene los datos *y* sus metadatos — es un formato "autodescriptivo" (el archivo explica su propio contenido).
* Soporta tipos de datos estándar y personalizados.
* Soporta filtros (checksums, compresión, escalado).
* Soporta entrada/salida paralela, e es independiente de la plataforma.
* Se lo considera un formato "a prueba de futuro" (seguirá siendo legible dentro de muchos años).
* Muy usado en HPC por códigos de simulación, herramientas de análisis y visualización (VisIt, ParaView, IDL, MATLAB).

### Estructura de HDF5

Un archivo HDF5 tiene una estructura de árbol con grupos, *datasets* (conjuntos de datos) y atributos — es una analogía razonable a un sistema de archivos con carpetas y archivos.

![HDF5 schematic](fig/hdf5_structure4.jpg)

## H5Py: bindings de HDF5 para Python

* Interfaz de alto nivel, fácil de usar, muy "pythónica":
  * Sintaxis de diccionario (acceder a un dataset por nombre).
  * Sintaxis de arreglos de NumPy (indexado, slicing, etc.).
  * Acceso a metadatos mediante el atributo `attrs`.
* Diseñado desde el inicio para integrarse bien con NumPy.
* También existe una interfaz de bajo nivel en Cython hacia la API en C de HDF5, para casos avanzados.

### Escribir un arreglo de NumPy a HDF5

In [ ]:
import numpy as np
import h5py

M = np.random.rand(128,128)

with h5py.File("data.hdf5", 'w') as fp:
    # un archivo HDF5 siempre tiene un grupo raiz, "/", que usamos aqui implicitamente
    dset = fp.create_dataset("random_matrix", data=M)
    # agregar metadatos (atributos) que describen el dataset, via el objeto 'attrs'
    dset.attrs["whatis"] = "Esta es una matriz de 128x128 generada aleatoriamente."

    # o, de forma mas directa:
    fp["random_matrix2"] = M
    fp["random_matrix2"].attrs["whatis"] = "Esta es otra matriz de 128x128 generada aleatoriamente."


Usa siempre `with ... as fp:` al trabajar con archivos HDF5 — así te aseguras de que el archivo se cierre y se escriba correctamente, incluso si ocurre un error en el medio.

### Acceder a un dataset de HDF5

In [ ]:
import numpy as np
import h5py

with h5py.File("data.hdf5", 'r') as fp:
    print("Claves disponibles:", tuple(fp.keys()))

    # obtener el dataset
    dset = fp["random_matrix"]
    print(type(dset), dset.shape, dset.dtype)

    # obtener los metadatos que vienen con el dataset
    for key,val in dset.attrs.items():
        print("{} : {}".format(key, val))

    # dset se puede usar de forma parecida a un arreglo de numpy
    elem = dset[0,0]     # indexado, slicing
    dsum = np.sum(dset)  # ojo: np.sum(dset), no dset.sum()

    # conversion explicita a arreglo de numpy
    M = dset[:]
    print(type(M), M.shape, M.dtype)
    msum = M.sum()

print("Las sumas coinciden:", dsum == msum)

**Nota importante:** mientras el archivo HDF5 está abierto (dentro del bloque `with`), `dset` es un objeto especial de h5py, no un arreglo de NumPy — los datos todavía viven en el archivo, no en memoria. Recién cuando haces `dset[:]` (o cualquier indexado) se cargan a memoria como un `ndarray` real. Esto es justo lo que permite trabajar con archivos HDF5 mucho más grandes que la memoria RAM disponible: puedes leer solo el pedazo que necesitas en cada momento.

**Actividad:** crea tu propio archivo HDF5 con un dataset llamado `"mis_datos"` que contenga un arreglo de NumPy de tu elección (por ejemplo, `np.arange(50)`), agrégale un atributo `"autor"` con tu nombre, y después ábrelo de nuevo en modo lectura para imprimir el dataset completo y el atributo.

In [ ]:
import h5py
import numpy as np

# Escribe tu archivo
with h5py.File("mi_archivo.hdf5", "w") as fp:
    dset = fp.create_dataset("mis_datos", data=np.arange(50))
    dset.attrs["autor"] = "escribe aqui tu nombre"

# Ahora leelo de vuelta
with h5py.File("mi_archivo.hdf5", "r") as fp:
    print(fp["mis_datos"][:])
    print(fp["mis_datos"].attrs["autor"])

### Acceso paralelo (adelanto)

Este ejemplo requiere `mpi4py` y ejecutarse con varios procesos MPI a la vez (por ejemplo `mpirun -n 4 python script.py`) — **no lo ejecutes directamente en esta celda de Jupyter**, solo léelo como referencia. Vas a volver a este patrón en el notebook de MPI, más adelante en el curso.

In [ ]:
# NO EJECUTAR EN JUPYTER: requiere `mpirun -n 4 python script.py`
# from mpi4py import MPI
# import h5py

# rank = MPI.COMM_WORLD.rank  # el ID del proceso (entero de 0 a 3, para una corrida de 4 procesos)

# f = h5py.File('parallel_test.hdf5', 'w', driver='mpio', comm=MPI.COMM_WORLD)

# dset = f.create_dataset('test', (4,), dtype='i')
# dset[rank] = rank

# f.close()

## Uso avanzado de HDF5 en sistemas HPC

* *Checkpointing* (guardar puntos de control) de simulaciones de larga duración:
  * Escribir el estado actual de la simulación a un archivo HDF5.
  * Poder reanudar la simulación leyendo ese estado de vuelta, en vez de empezar desde cero si algo falla.
* Entrada/salida paralela a gran escala desde múltiples procesos MPI:
  * Escribir pocos archivos grandes por proceso (recuerda: limita la cantidad de archivos por carpeta, idealmente no más de ~1000, ver notebook 03).
  * O, mejor aún, escribir a un único archivo compartido usando una versión de `h5py` con soporte MPI.

## Resumen de I/O de HDF5

* Fácil de usar con `h5py` (una alternativa sería el módulo `pytables`).
* La estructura jerárquica permite guardar datos complejos en un único archivo.
* Los archivos HDF5 son portables y están pensados para durar en el tiempo.
* Los datos se pueden compartir y usar desde otros programas y lenguajes.

$\to$ Esta es la recomendación general del curso para I/O relacionada con NumPy.

# Aplicación: procesamiento de imágenes

## Ejemplo 1: suavizar imágenes

![image filter](fig/image-filter-example.svg)

**Objetivo:** quitar ruido de una imagen. Para cada píxel en la posición $(i, j)$, calculamos:

$$\tilde{f}_{i,j} = \alpha f_{i,j} + \frac{(1 - \alpha)}{4} (f_{i-1,j} + f_{i+1,j} + f_{i,j-1} + f_{i,j+1})$$

Es decir: mezclamos el valor del píxel con el promedio de sus 4 vecinos — esto filtra el "ruido de alta frecuencia" (variaciones bruscas pixel a pixel). Con $\alpha = 0$ el suavizado es máximo; con $\alpha = 1$ no hay suavizado (la imagen queda igual). ¿Te suena familiar? Es la misma idea que la ecuación de difusión del teaser: cada punto se actualiza mezclándose con sus vecinos.

### La imagen: Alfa Centauri

Imagen del DSS (*Digital Sky Survey* — ver [términos de uso](http://archive.stsci.edu/dss/acknowledging.html)).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# convertir a flotante, entre 0 y 1
image = plt.imread("fig/dss_alpha_centauri.gif").astype(np.float64)/255.
plt.imshow(image, cmap='gray')
print("Forma: ", image.shape)

## Loops en Python vs. arreglos de NumPy

Igual que en el teaser de difusión: misma fórmula, dos formas de escribirla.

In [ ]:
def get_filtered_loop(image, alpha=0.25):
    filtered = np.empty_like(image)
    N, M = image.shape
    for i in range(1, N-1):
        for j in range(1, M-1):
            filtered[i, j] = 0.25*(1.0-alpha)*(image[i-1, j] + image[i+1, j]
                                               + image[i, j-1] + image[i, j+1]) \
                + alpha*image[i, j]
    return filtered
%timeit get_filtered_loop(image)

In [ ]:
def get_filtered_array(image, alpha=0.25):
    filtered = np.empty_like(image)
    N, M = image.shape
    filtered[1:N-1, 1:M-1] = 0.25*(1.0-alpha)*(image[0:N-2, 1:M-1] + image[2:N, 1:M-1]
                                               + image[1:N-1, 0:M-2] + image[1:N-1, 2:M]) \
        + alpha*image[1:N-1, 1:M-1]
    return filtered
%timeit get_filtered_array(image)

**Tus resultados:** compara los dos tiempos — deberías ver, otra vez, un factor de entre 10 y 100 más rápido para la versión vectorizada.

### La transformación, paso a paso

* El loop `for i in range(1, N-1): for j in range(1, M-1):` se traduce, del lado izquierdo, a `filtered[1:N-1, 1:M-1]`.
* Los índices del lado derecho se desplazan: `i-1` → `0:N-2`, `i` → `1:N-1`, `i+1` → `2:N` (y lo mismo para `j`).
* Por ejemplo: `image[i-1, j]` se convierte en `image[0:N-2, 1:M-1]`.

Es exactamente la misma receta que usamos para el ejemplo de puntos medios más arriba, solo que ahora en 2 dimensiones en vez de 1.

In [ ]:
filtered_loop = get_filtered_loop(image)[1:-1,1:-1]
filtered_array = get_filtered_array(image)[1:-1,1:-1]

f, axs = plt.subplots(2, 3, figsize=(14, 8))
axs[0, 0].imshow(image, cmap='gray')
axs[0, 1].imshow(filtered_array, cmap='gray')
axs[0, 2].imshow(filtered_array - filtered_loop, cmap='gray')
axs[1, 0].imshow(image[600:700, 600:700], cmap='gray')
axs[1, 1].imshow(filtered_array[600:700, 600:700], cmap='gray')
axs[1, 2].imshow((image[1:-1,1:-1]-filtered_array)[600:700, 600:700], cmap='gray')
print("El metodo de loop y el de arreglos dan el mismo resultado: ", np.allclose(filtered_loop, filtered_array))

La fila de arriba muestra la imagen completa (original, suavizada, y la diferencia entre ambas versiones — que debería verse toda negra, porque dan el mismo resultado). La fila de abajo hace zoom sobre una región de 100x100 píxeles para que se note mejor el efecto del suavizado.

**Actividad:** vuelve a la celda de `get_filtered_array` y cambia `alpha` de `0.25` a `0.05`, y después a `0.8`. Vuelve a generar el gráfico de comparación (la celda de `f, axs = plt.subplots(...)`) cada vez. ¿Qué le pasa a la imagen suavizada en cada caso? Relaciónalo con la fórmula: ¿qué significa un `alpha` cercano a 0 o cercano a 1?

## Ejemplo 2: detectar bordes

Usamos esta fórmula (parecida a la anterior, pero ahora restando el promedio de los vecinos en vez de mezclarlo, y usando también las diagonales):

$$\tilde{f}_{i,j} = 8 f_{i,j} - (f_{i-1,j} + f_{i+1,j} + f_{i,j-1} + f_{i,j+1} + f_{i-1,j-1} + f_{i-1,j+1} + f_{i+1,j+1} + f_{i+1,j-1})$$

Esto resalta las zonas donde el valor de un píxel difiere mucho de sus 8 vecinos (bordes y contornos), y después recortamos el resultado a un cierto percentil para que el contraste se vea mejor.

In [ ]:
from scipy.datasets import ascent
im = plt.imshow(ascent(), cmap='gray')

### Nuestra implementación

In [ ]:
def get_edges(image, percentiles=(25, 99)):
    result = np.empty_like(image)
    N, M = image.shape
    result[1:N-1, 1:M-1] = -1.0*(image[0:N-2, 1:M-1] + image[2:N, 1:M-1]
                                 + image[1:N-1, 0:M-2] + image[1:N-1, 2:M]
                                 + image[0:N-2, 0:M-2] + image[0:N-2, 2:M]
                                 + image[2:N, 0:M-2] + image[2:N, 2:M])     + 8.*image[1:N-1, 1:M-1]
    # recortar el resultado a un rango de percentiles, para mejorar el contraste visual
    fmin, fmax = np.percentile(result, percentiles)
    result[result < fmin] = fmin
    result[result > fmax] = fmax
    return result
f, axs = plt.subplots(1, 2, figsize=(15, 5))
axs[0].imshow(ascent(), cmap='gray')
axs[1].imshow(get_edges(ascent().astype('int32')), cmap='gray')

**Actividad:** llama a `get_edges(ascent().astype('int32'), percentiles=(5, 95))` y despliega el resultado con `plt.imshow`. Compara contra el resultado con los percentiles por defecto `(25, 99)`. ¿Qué efecto tiene ampliar o achicar el rango de percentiles sobre el contraste de la imagen?

In [ ]:
resultado_alterno = get_edges(ascent().astype('int32'), percentiles=(5, 95))
plt.imshow(resultado_alterno, cmap='gray')

### Comparación con un filtro predefinido

No hace falta reinventar la rueda cada vez: librerías de procesamiento de imágenes como Pillow (`PIL`) ya traen filtros de detección de bordes listos para usar. Comparamos nuestra implementación casera contra el filtro `FIND_EDGES` de Pillow.

In [ ]:
from PIL import Image, ImageFilter
img = Image.fromarray(ascent().astype("int32"), mode="I").convert("L")
filtered_img = img.filter(ImageFilter.FIND_EDGES)

f, axs = plt.subplots(1, 2, figsize=(15, 5))
axs[0].imshow(ascent(), cmap='gray')
axs[1].imshow(np.asarray(filtered_img), cmap='gray')

## Para cerrar

Este bonus de procesamiento de imágenes es, en el fondo, el mismo patrón que viste en toda la notebook: una fórmula que actualiza cada elemento en función de sus vecinos, escrita primero con loops explícitos y después vectorizada con slicing de NumPy. Es el mismo patrón que vas a reencontrar en el caso de estudio de difusión 2D (notebook 09) y en el ejercicio de advección — vale la pena que te quede bien claro antes de seguir avanzando en el curso.